In [1]:
# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys
import ctypes

try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
except Exception:
    pass

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"
ARTIFACTS_DIR = f"{codebase_path}/artifacts"
print("💻 Local Lab Server Environment Loaded.")

💻 Local Lab Server Environment Loaded.


## 1. Load Data & Base Model

In [2]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from src.inference.evaluate import run_evaluation

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

val_full = load_jsonl(f"{DATA_DIR}/val_full_info.jsonl")
val_struct = load_jsonl(f"{DATA_DIR}/val_structural.jsonl")
print(f"Loaded {len(val_full)} full-info validation samples and {len(val_struct)} structural validation samples.")


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loaded 2039 full-info validation samples and 2039 structural validation samples.


In [3]:
# === CONFIGURATION ===
MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"

COMPUTE_DTYPE = torch.bfloat16

SFT_FULL_DIR = f"{MODELS_DIR}/sft_fullInfo_Qwen2.5-32B-Instruct"
# SFT_STRUCT_DIR = f"{MODELS_DIR}/sft_structOnly_Qwen2.5-32B-Instruct"

In [4]:
# === LOAD BASE MODEL ===
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=COMPUTE_DTYPE,
)
base_model.eval()
print("Base model loaded successfully!")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Base model loaded successfully!


## 2. Evaluate SFT Full-Information Model

In [5]:
# Load the Full-Info LoRA adapter
try:
    print(f"Loading adapter from {SFT_FULL_DIR}...")
    model_full = PeftModel.from_pretrained(base_model, SFT_FULL_DIR)
    
    # Evaluate on Full Info Dataset
    acc_sft_full, results_sft_full = run_evaluation(
        model=model_full,
        tokenizer=tokenizer,
        dataset=val_full,
        method_name="SFT_LoRA",
        config_name="FullInfo",
        model_name=MODEL_ID,
        output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
    )
    
    # Unload adapter to free memory for the next evaluation
    model_full.unload()
except Exception as e:
    print(f"Could not load or evaluate Full-Info SFT model: {e}")


Loading adapter from ..//output/models/sft_fullInfo_Qwen2.5-32B-Instruct...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating SFT_LoRA - FullInfo:   0%|          | 0/2039 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

Evaluating SFT_LoRA - FullInfo: 100%|█████████▉| 2037/2039 [41:20<00:02,  1.16s/it]

Evaluating SFT_LoRA - FullInfo: 100%|█████████▉| 2038/2039 [41:21<00:01,  1.17s/it]

Evaluating SFT_LoRA - FullInfo: 100%|██████████| 2039/2039 [41:22<00:00,  1.21s/it]

Evaluating SFT_LoRA - FullInfo: 100%|██████████| 2039/2039 [41:22<00:00,  1.22s/it]

\n--- Evaluation Results ---
Method: SFT_LoRA
Config: FullInfo
Model: Qwen/Qwen2.5-32B-Instruct
Accuracy: 0.9735
Format Error Rate: 0.0010
Semantic Confusion: 0.5863
Option Bias (A): 0.1422
Latency: 2482.61 seconds


## 3. Evaluate SFT Structural-Only Model

In [6]:
# # Load the Structural-Only LoRA adapter
# try:
#     print(f"Loading adapter from {SFT_STRUCT_DIR}...")
#     model_struct = PeftModel.from_pretrained(base_model, SFT_STRUCT_DIR)
    
#     # Evaluate on Structural Only Dataset
#     acc_sft_struct, results_sft_struct = run_evaluation(
#         model=model_struct,
#         tokenizer=tokenizer,
#         dataset=val_struct,
#         method_name="SFT_LoRA",
#         config_name="StructuralOnly",
#         model_name=MODEL_ID,
#         output_csv=f"{ARTIFACTS_DIR}/experiment_summary.csv"
#     )
    
#     # Unload adapter
#     model_struct.unload()
# except Exception as e:
#     print(f"Could not load or evaluate Structural-Only SFT model: {e}")
